Python
markdown_content = """# 🌐 분산 시뮬레이션 상자 동기화 전담 노드(sim_sync_node) 구축 계획서

> [!IMPORTANT]
> **AI 에이전트 및 개발자 가이드**: 본 문서는 단일 PC의 리소스 한계를 극복하고 Isaac Sim 구동 프레임(FPS)을 최적화하기 위해, **bg2(분류 라인)**와 **sg2(적재/포장 라인)** 환경 간의 상자 순간이동(소멸 및 소환) 로직을 전담 제어하는 독립 노드(`sim_sync_node`)의 설계 및 구현 명세서입니다.

---

## 📌 1. 독립 노드 분리 설계 배경 (Architectural Decision)

기존 관제탑(`control_tower_node.py`)은 데이터베이스 커넥션 풀링, AMR 큐 스케줄링, 로봇 팔 일시정지 제어 등 핵심 비즈니스 로직만으로도 처리해야 할 결합도가 매우 높습니다. 
여기에 시뮬레이터(Isaac Sim) 두 인스턴스 간의 물리적 프림(Prim) 상태를 강제로 동기화하는 '소멸/소환' 이벤트 핸들러까지 추가되면 다음과 같은 문제가 발생합니다:

1. **관제탑 비대화(Fat Node):** 시뮬레이션 종속적 코드가 메인 관제 로직에 섞여 유지보수성이 저하됩니다.
2. **상용화/현업 전환 시 결함:** 실제 현장 물류창고에 배포할 때는 시뮬레이터 동기화 코드를 전부 주석 처리하거나 지워야 하는 번거로움이 생깁니다.

따라서 완전히 독립된 **마이크로 ROS 2 서비스 노드(`sim_sync_node.py`)**로 기능을 분리하여, 개발 환경(시뮬레이션 가속)과 상용 환경의 결합도를 원천 차단(Decoupling)합니다.

---

## 🔄 2. 데이터 흐름 및 상자 텔레포트 메커니즘

두 대의 독립된 Isaac Sim 인스턴스는 서로 연결되어 있지 않으며, 오직 `sim_sync_node`가 발행하는 ROS 2 통신망과 하이브리드 DB 상태 캐시를 기점으로 핑퐁 연동을 수행합니다.

코드 출력
File SIM_SYNCHRONIZATION_NODE_PLAN.md generated successfully.

```mermaid
sequenceDiagram
    autonumber
    participant Isaac_bg2 as Isaac Sim A (bg2)
    participant SyncNode as ROS 2 sim_sync_node
    participant DB as PostgreSQL / Redis
    participant Isaac_sg2 as Isaac Sim B (sg2)

    Isaac_bg2->>Isaac_bg2: 상자 컨베이어 끝 트리거 박스 접촉 감지
    Isaac_bg2->>SyncNode: [ServiceCall] /sim/transit_package (ID, 대상 라인)
    activate SyncNode
    SyncNode->>DB: UPDATE packages SET status = 'TRANSIT_TO_SG2'
    SyncNode->>Isaac_sg2: [Topic Pub] /sim/sg2_spawn_trigger (JSON payload)
    SyncNode-->>Isaac_bg2: Response (success = true)
    deactivate SyncNode
    Isaac_bg2->>Isaac_bg2: 월드 내 해당 상자 Prim 즉시 삭제 (소멸)
    
    Isaac_sg2->>Isaac_sg2: Topic 구독 감지 후 해당 입구 좌표에 Prim 동적 생성 (소환)
💻 3. 전담 동기화 노드 소스 코드 명세 (sim_sync_node.py)
이 노드는 관제탑과 독립적으로 컴파일 및 가동되며, 데이터베이스 상태 동기화와 소환 이벤트 전달만 초고속 비동기로 처리합니다. (기존 데이터베이스 연결 계정을 공유)

Python
#!/usr/bin/env python3
import rclpy
from rclpy.node import Node
from std_msgs.msg import String
from cobot3_interfaces.srv import ReportInboundProgress # 또는 커스텀 서비스 호환 사용
import psycopg2
import redis
import json
import os
import time

class SimSyncNode(Node):
    def __init__(self):
        super().__init__('sim_sync_node')
        self.get_logger().info('=== 🌐 분산 시뮬레이션 상자 동기화 전담 노드(SimSync) 구동 ===')

        # 1. DB 및 Redis 환경 변수 획득
        pg_host = os.environ.get('POSTGRES_HOST', 'localhost')
        redis_host = os.environ.get('REDIS_HOST', 'localhost')

        try:
            self.conn = psycopg2.connect(
                host=pg_host, port=5432, user='rokey', password='rokey_pass', database='warehouse_db'
            )
            self.conn.autocommit = True
            self.redis_client = redis.Redis(host=redis_host, port=6379, decode_responses=True)
            self.get_logger().info('PostgreSQL 및 Redis 캐시 시스템 연동 완료.')
        except Exception as e:
            self.get_logger().error(f'인프라 연결 실패: {str(e)}')

        # 2. bg2 시뮬레이터로부터 '탈출 신호'를 수신할 ROS 2 토픽/서비스 정의
        # 확장성과 범용성을 위해 JSON 직렬화 문자열 토픽 채널 개설
        self.bg2_exit_sub = self.create_subscription(
            String,
            '/sim/bg2_exit_event',
            self.bg2_exit_callback,
            10
        )

        # 3. sg2 시뮬레이터에게 '소환 명령'을 하달할 ROS 2 퍼블리셔 정의
        self.sg2_spawn_pub = self.create_publisher(String, '/sim/sg2_spawn_trigger', 10)

    def bg2_exit_callback(self, msg):
        \"\"\"bg2 벨트 끝단 접촉 시 트리거되는 코어 비동기 콜백\"\"\"
        try:
            data = json.loads(msg.data)
            package_id = data.get('package_id')
            target_line = data.get('target_line') # 예: 'sg2_in_01', 'sg2_in_02'

            self.get_logger().info(f'[싱크 통신] 상자 탈출 감지 ➔ ID: {package_id} | 목적라인: {target_line}')

            # 1. PostgreSQL DB 상태 마스킹 (이동 중 상태로 변환하여 데이터 정합성 보존)
            cursor = self.conn.cursor()
            cursor.execute(
                "UPDATE packages SET status = 'TRANSIT_TO_SG2' WHERE package_id = %s;",
                (package_id,)
            )
            cursor.close()

            # 2. sg2 시뮬레이터 PC가 구독 중인 채널로 소환 명령 던지기
            spawn_payload = {
                "package_id": package_id,
                "target_line": target_line,
                "timestamp": time.time()
            }
            
            pub_msg = String()
            pub_msg.data = json.dumps(spawn_payload)
            self.sg2_spawn_pub.publish(pub_msg)
            self.get_logger().info(f'[싱크 통신] sg2 월드로 소환 이벤트 송신 완료 ➔ {package_id}')

        except Exception as e:
            self.get_logger().error(f'동기화 처리 루프 에러: {str(e)}')

def main(args=None):
    rclpy.init(args=args)
    node = SimSyncNode()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        if hasattr(node, 'conn'):
            node.conn.close()
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()
🛠️ 4. Isaac Sim 사이드 스크립트 연동 가이드 (Hooks)
① Isaac Sim A (bg2) - 벨트 끝 단에서 호출할 Hook 예시
상자가 오늘/내일/모레 분류 슈트(Chute) 컨베이어 트리거 영역에 충돌했을 때, ROS 2 통신망으로 노드에 노크를 보냅니다.

Python
import omni.isaac.core.utils.prims as prim_utils
import rclpy
from std_msgs.msg import String
import json

# ROS 2 퍼블리셔가 사전 초기화되어 있다고 가정
def on_box_exit_chute(package_id, target_line, prim_path):
    # 1. 독립 노드에 비동기 전달용 페이로드 직렬화
    exit_data = {"package_id": package_id, "target_line": target_line}
    msg = String()
    msg.data = json.dumps(exit_data)
    bg2_exit_pub.publish(msg)
    
    # 2. 내 월드 내비게이션 및 물리 부하 해소를 위해 즉시 삭제
    if prim_utils.is_prim_path_valid(prim_path):
        prim_utils.delete_prim(prim_path)
② Isaac Sim B (sg2) - 백그라운드 소환 처리 Hook 예시
sim_sync_node가 퍼블리시한 토픽을 실시간으로 캐치하여 물리 월드 내 지정된 컨베이어 진입로에 상자를 실시간 투하합니다.

Python
from omni.isaac.core.objects import DynamicCuboid
from omni.isaac.core.utils.string import find_unique_path

# 라인별 입구 물리 좌표 매핑 (PHYSICAL_LAYOUT.md 규격 준수)
LINE_ENTRY_COORDS = {
    'sg2_in_01': [7.5, 3.0, 0.4], # 컨베이어 진입로 X, Y, Z 절대좌표
    'sg2_in_02': [7.5, -1.5, 0.4],
    'sg2_in_03': [7.5, -6.0, 0.4]
}

def sg2_spawn_trigger_callback(msg):
    data = json.loads(msg.data)
    pkg_id = data['package_id']
    line = data['target_line']
    
    pos = LINE_ENTRY_COORDS.get(line, [7.5, 3.0, 0.4])
    prim_path = find_unique_path(f"/World/Packages/{pkg_id}")
    
    # sg2 물리 환경에 정육면체 동적 스폰
    new_box = DynamicCuboid(
        prim_path=prim_path,
        name=pkg_id,
        position=pos,
        scale=[0.35, 0.35, 0.35], # 상자 크기 최적화
        color=[0.0, 0.5, 1.0]     # sg2에 소환된 상자는 파란색으로 식별 유도
    )
    print(f"📦 [sg2 월드] 상자 {pkg_id}가 {line} 진입로에 성공적으로 소환되었습니다.")
🚀 5. 통합 기동 가이드 (start_test_env.sh 연동)
원클릭 가동 스크립트에 이 마이크로 서비스를 추가하여 일괄 제어가 가능하도록 변경합니다.

Bash
# start_test_env.sh 에 아래 구문을 관제탑 실행 구문 옆에 추가해 줍니다.
echo "🌐 분산 시뮬레이션 상자 동기화 노드 백그라운드 실행..."
ros2 run cobot3 sim_sync_node > /dev/null 2>&1 &
📊 6. 동적 풀링 관제와 결합 시 기대 효과
독립성 강화: 시뮬레이션 관련 동기화 코드가 분리되어, control_tower_node.py 파일은 오직 순수 대형 AGV Fleet 배정(Dynamic Pooling)과 트래픽 데드락 제어 규칙에만 온전히 컴퓨팅 자원을 할당할 수 있습니다.

추적 신뢰성: 상자가 컴퓨터망을 타고 움직이는 동안 상태가 TRANSIT_TO_SG2로 대시보드에 표기되므로 시각적인 모니터링 품질이 비약적으로 증가합니다.
"""

file_name = "SIM_SYNCHRONIZATION_NODE_PLAN.md"
with open(file_name, "w", encoding="utf-8") as f:
f.write(markdown_content)

print(f"File {file_name} generated successfully.")

Your Markdown file is ready
[file-tag: code-generated-file-0-1781073085915974672]

구상하신 **분산 시뮬레이션 환경(2대 아이작 심 분리 구동)에서의 상자 순간이동(소멸 및 소환)** 프로세스를 메인 관제탑 노드에 섞지 않고, 소프트웨어 공학적으로 깔끔하게 격리한 **독립 전담 노드 구축 계획서 및 구현 소스 코드 명세**를 Markdown 파일로 생성 완료했습니다.

### 📝 생성된 문서의 구조 및 핵심 요약

1. **독립 노드 분리 배경 (Architectural Decision)**:
   * 메인 관제탑(`control_tower_node.py`)이 비대해지는 현상(Fat Node)을 방지하고 비즈니스 로직(AMR 동적 풀링 스케줄링)의 안전성을 완벽히 보호하기 위한 결합도 차단(Decoupling) 전략 기술.
2. **상자 텔레포트 데이터 흐름 다이어그램 (Sequence 명세)**:
   * 상자가 `bg2` 벨트 끝에 닿는 순간부터 ➔ 독립 노드가 캐치하여 PostgreSQL 상자 상태를 `TRANSIT_TO_SG2`(이송 중)로 갱신 ➔ `sg2` 월드로 소환 명령 토픽을 쏘기까지의 파이프라인 정립.
3. **`sim_sync_node.py` 소스 코드 수록**:
   * 그대로 복사해서 새 파일로 컴파일 및 가동할 수 있는 든든한 파이썬 기반 ROS 2 마이크로 서비스 노드 풀 소스 코드 포함.
4. **시뮬레이터(Isaac Sim) 양대 사이드 Hook 가이드**:
   * `bg2` 시뮬레이터에서 상자 프림을 지우며 관제망에 노크하는 훅 코드.
   * `sg2` 시뮬레이터에서 신호를 받아 `PHYSICAL_LAYOUT.md` 좌표 규격에 맞춰 상자를 실시간 투하(`Spawn`)하는 스폰 처리 훅 코드 수록.
5. **통합 기동 연동 (`start_test_env.sh`)**:
   * 백그라운드 런처 스크립트에 통합하는 쉘 명령어 안내.

나중에 이 구조를 다시 확인하더라도 전체 통신 아키텍처와 시뮬레이터 내부 스크립트 작성 방향을 한눈에 바로 파악하실 수 있도록 꼼꼼히 구성해 두었습니다. 

이 전담 동기화 노드 구성을 토대로 실제 ROS 2 패키지 디렉토리에 소스코드를 추가하는 작업을 이어서 진행해 볼까요? 궁금하거나 보완이 필요한 점이 있다면 말씀해 주세요!